# MFQDMA — reproduce every figure and table of Paper I

**S. Pratiher**, *Quantile detrending moving average algorithm for multifractal time series and surfaces*, (2026).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<YOUR-GITHUB-USERNAME>/MFQDMA-PaperI/blob/main/MFQDMA_PaperI.ipynb)

This single notebook regenerates **all 17 figures and all 11 data tables** of the paper.

| Part | What it does | Run time (one core) |
|---|---|---|
| 0. Setup | clones the repository (Colab), installs the requirements | < 1 min |
| 1. Recompute (optional) | re-runs the numerical experiments and overwrites `results/` | off by default; 5 min – 1 h per group, see the flags |
| 2. Figures 1–17 | draws every figure from `results/` (Figs. 1–4, 7, 17 are computed directly) | about 10 min |
| 3. Tables II–XII | generates every data table from `results/` (LaTeX rows in `tables_out/`) | seconds |
| 4. Check | compares the regenerated figures with the manuscript figures in `figures/` | seconds |
| 5. Download | zips `figures_out/`, `tables_out/` and `results/` | seconds |

By default the notebook uses the stored results in `results/`, so **Runtime → Run all** reproduces the paper in about ten minutes. To regenerate the results themselves from scratch, switch on the flags in Part 1. All random seeds are fixed, so the recomputed results reproduce the stored ones.

**Data.** Synthetic data are generated in the code. Daily prices come from the CRAN package *qrmdata* (Hofert, Hornik & McNeil), which is cloned automatically from GitHub and read by a pure-Python parser (R is not needed). The terrain model ships with Matplotlib. The Bonn EEG database (Andrzejak *et al.*, Phys. Rev. E **64**, 061907, 2001) is needed only to *recompute* the EEG results (Part 1).

Companion paper (cross-correlations, MF-X-QDMA): repository `MF-X-QDMA`.

## 0. Setup

In [ ]:
#@title Setup (run first): clone the repository in Colab and install requirements
import os, sys, subprocess
GITHUB_REPO = 'https://github.com/Sawon1234/MFQDMA/MFQDMA_main.git'   # <- edit after upload
SUBDIR = ''          # path of this folder inside the repository, if it is not the repository root

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    root = '/content/' + GITHUB_REPO.rstrip('/').split('/')[-1].replace('.git', '')
    if not os.path.isdir(root):
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, root], check=True)
    os.chdir(os.path.join(root, SUBDIR))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
if not os.path.isdir('mfqdma'):
    raise RuntimeError('Run this notebook from the folder that contains MFQDMA_PaperI.ipynb and mfqdma/ '
                       '(in Colab: set GITHUB_REPO and SUBDIR above).')
sys.path.insert(0, os.getcwd())
try:
    from IPython.display import display, Markdown
except ImportError:                      # plain Python (no Jupyter): print instead
    display = print
    Markdown = str
import time, json, numpy as np, pandas as pd, matplotlib
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
print('working directory:', os.getcwd(), '| numpy', np.__version__, '| matplotlib', matplotlib.__version__)

## 1. Recompute the stored results (optional)

Each flag re-runs one group of experiments and overwrites the corresponding file in `results/`. Leave everything `False` to reproduce the paper from the stored results. Run times are rough estimates for a free Colab runtime (measured on one CPU core and rounded up). With `RECOMPUTE_ALL = True` the whole study takes about 2 h.

In [ ]:
RECOMPUTE_ALL = False
RECOMPUTE = dict(
    corpus_1d   = False,   # results/bench_1d.json       Tables II, III             ~10 min
    advantage   = False,   # results/advantage_all.json  Table IV, Figs. 5, 6       ~45 min
    analysis_2d = False,   # results/analysis_2d_all.json Table V, Figs. 8, 9       ~25 min
    mono_2d     = False,   # results/paper_2d_mono.json  Table VI, Fig. 9           ~5 min
    finance     = False,   # results/finance.json        Tables VII, VIII, Figs. 10-13 ~15 min
    dem         = False,   # results/dem.json            Table XII                  ~5 min
    eeg         = False,   # results/real_eeg.json, eeg_main.npz, eeg_auc.json, eeg_preproc.json
                           #                             Tables IX-XI, Figs. 14-16  ~10 min, needs the Bonn data
)
if RECOMPUTE_ALL:
    RECOMPUTE = {k: True for k in RECOMPUTE}
from mfqdma import experiments_paperI as E
print({k: v for k, v in RECOMPUTE.items() if v} or 'using the stored results in results/')

### 1a. Synthetic benchmarks

In [ ]:
if RECOMPUTE['corpus_1d']:
    E.run_corpus_1d()        # 8 binomial measures, 18 fGn series, 11 estimators
if RECOMPUTE['advantage']:
    E.run_advantage()        # contamination, alpha-stable noise, monotone maps, asymmetric cascades
if RECOMPUTE['analysis_2d']:
    E.run_analysis_2d()      # 2D cascades (theta x lambda), raw fBm surfaces, contaminated 2D cascades
if RECOMPUTE['mono_2d']:
    E.run_2d_mono()          # mixed increments of fractional Brownian surfaces

### 1b. Financial markets and terrain

In [ ]:
if RECOMPUTE['finance']:
    E.run_finance()          # 18 assets + 29 DJIA stocks; clones github.com/cran/qrmdata on first use
if RECOMPUTE['dem']:
    E.run_dem()              # terrain elevation model and its surrogates

### 1c. Bonn EEG (only if `RECOMPUTE['eeg']`)

The Bonn EEG database is not redistributed here. Download the five sets Z, O, N, F, S (100 text files each) from the University of Bonn epileptology department (see Andrzejak *et al.* 2001), then point `BONN_SOURCE` at them. It may be a folder, a list of the five zip files, or one archive such as `Dataset.rar`. In Colab, upload the files with the folder icon on the left or mount Google Drive.

In [ ]:
BONN_SOURCE = []   # e.g. ['/content/drive/MyDrive/Dataset.rar'] or ['z.zip', 'o.zip', 'n.zip', 'f.zip', 's.zip']
if RECOMPUTE['eeg']:
    from mfqdma import paths
    if not (paths.BONN / 'Z').is_dir():
        if IN_COLAB and any(str(s).lower().endswith('.rar') for s in BONN_SOURCE):
            subprocess.run('apt-get -qq install -y unrar > /dev/null', shell=True)
        paths.prepare_bonn(*BONN_SOURCE)
    E.run_eeg_preprocessing()    # Table IX
    E.run_eeg_surrogates()       # Table X, Fig. 14
    E.run_eeg_main()             # Figs. 15, 16
    E.run_eeg_classification()   # Table XI, Fig. 16(d)

## 2. Figures

Each cell draws one figure of the paper, shows it, and writes it to `figures_out/` as PDF and PNG. The plotting code is in `mfqdma/figures_paperI.py` (one function per figure). Figures 1–4, 7 and 17 are computed directly; their intermediate results are cached in `cache/`.

In [ ]:
from mfqdma import figures_paperI as F
from mfqdma import paths
for p in paths.CACHE.glob('*.npy'):   # fresh start: recompute the direct computations
    p.unlink()

### Fig. 1. The 1D MFQDMA algorithm

The 1D MFQDMA algorithm on an fGn with $H=0.7$ ($N=2^{14}$, $\theta=0$, $n=128$). (a) Profile $y(i)$ with the moving mean of MFDMA and the moving quantiles $\tilde y_\lambda$ for $\lambda=0.1$, $0.5$, $0.9$. (b) Residuals $\epsilon(i)$ of MFDMA and $\epsilon_\lambda(i)$ for $\lambda=0.1$ and $0.9$. (c) $F_2(n,\lambda)$ for MFDMA and five quantile levels; the slopes $h(2,\lambda)$ are given in the legend.

*Source:* Computed directly (a few seconds).

In [ ]:
F.make_figure(1)

### Fig. 2. Binomial measure

Binomial measure, $p=0.3$, $N=2^{16}$. (a) $F_q(n)$ of backward MFQDMA ($\lambda=0.25$) for $q=-4,-2,0,2,4$, shifted vertically for clarity; lines are least-squares fits. (b) $\tau(q)$ from MFDFA, backward MFDMA, and backward MFQDMA versus theory. (c) $\Delta\tau(q)$ for all methods and five quantile levels. (d) Singularity spectra versus theory.

*Source:* Computed directly (a few seconds).

In [ ]:
F.make_figure(2)

### Fig. 3. Gliding-box representation and degeneracy line

Proposition 1 and the degeneracy line. (a) A monotone profile of a binomial measure ($p=0.3$) with a backward window (shaded, $n=33$). The $\lambda=0.25$ quantile (dashed) equals $y(i-\ell)$, so $\epsilon_\lambda(i)$ is the box mass $\mu(i-\ell,i]$. (b) Standard deviation of the residual versus $\theta$ ($n=65$), normalized by that of backward MFDMA. Dotted lines mark $\theta=1-\lambda$. (c) $\mathrm{RMSE}_\tau$ versus $\theta$ for $N=2^{16}$.

*Source:* Computed directly (about 2 min; cached in `cache/`).

In [ ]:
F.make_figure(3)

### Fig. 4. λ independence for measures; monofractal fGn

(a) Corollary 2: $h(q,\lambda)$ of a binomial measure ($p=0.3$, $N=2^{16}$) for five quantile levels, compared with MFDMA and with theory. (b) $h(2)$ versus $H$ for fGn (mean $\pm$ s.d. of three realizations, $N=2^{16}$); the line is $h=H$. (c) Spurious width $\max_qh(q)-\min_qh(q)$ for fGn.

*Source:* Computed directly (18 fGn series of length $2^{16}$, about 3 min; cached in `cache/`).

In [ ]:
F.make_figure(4)

### Fig. 5. Contaminated binomial measure

Robustness of $\tau(q)$ estimates of a contaminated binomial measure ($p=0.3$, $N=2^{16}$). (a)--(c) $\mathrm{RMSE}_\tau$ versus the contaminated fraction $\varepsilon$ for three contamination types (mean of five replicates). (d) Ratio $\mathrm{RMSE}_\tau(\mathrm{MFQDMA})/\mathrm{RMSE}_\tau(\mathrm{MFDMA})$ for $\lambda=0.1$, $0.25$, $0.9$, paired by replicate; values below the horizontal line favor MFQDMA.

*Source:* Reads `advantage_all.json` in `results/`.

In [ ]:
F.make_figure(5)

### Fig. 6. Negative results (α-stable noise, monotone maps)

Negative results. (a) Spurious width of $h(q)$ for independent symmetric $\alpha$-stable noise ($N=2^{15}$, five realizations) versus the stable index. (b) Mean absolute change of $h(q)$ of a log-normal cascade under strictly increasing maps. Bar order in each group: MFDFA, MFDMA, and MFQDMA with $\lambda=0.1$, $0.25$, $0.5$, $0.75$, $0.9$.

*Source:* Reads `advantage_all.json` in `results/`.

In [ ]:
F.make_figure(6)

### Fig. 7. 2D four-multiplier cascade

2D four-multiplier cascade, set A, $512\times512$. (a) Measure normalized by its mean (logarithmic color scale). (b) $F_q(n)$ of backward 2D-MFQDMA ($\lambda=0.25$), shifted vertically. (c) $\Delta\tau(q)$ for backward 2D-MFDMA and 2D-MFQDMA. (d) $f(\alpha)$ versus theory. For this realization $\mathrm{RMSE}_\tau=0.087$ (MFDMA), $0.099$ ($\lambda=0.25$), and $0.084$ ($\lambda=0.5$).

*Source:* Computed directly ($512\times512$ quantile trend, about 2 min).

In [ ]:
F.make_figure(7)

### Fig. 8. 2D: window position and contamination

(a) $\mathrm{RMSE}_\tau$ of $256^2$ cascades (mean over sets A--C) for all combinations of trend functional and window position. (b), (c) $\mathrm{RMSE}_\tau$ of the set-A cascade ($256^2$) when a fraction $\varepsilon$ of pixels is replaced by $\pm20\sigma$ outliers or Student-$t_2$ values (mean of two replicates).

*Source:* Reads `analysis_2d_all.json` in `results/`.

In [ ]:
F.make_figure(8)

### Fig. 9. Fractional Brownian surfaces

Fractional Brownian surfaces ($256^2$). (a) $h(2)$ versus $H$ from the raw surface $X$ (dashed) and from its mixed increments $\Delta_1\Delta_2X$ (solid, mean $\pm$ s.d.); the black line is $h=H$. (b) $h(q)$ of the increments for $H=0.3$ (solid) and $H=0.7$ (dashed); horizontal lines mark $H$. (c) $F_q(n)$ of 2D-MFQDMA ($\lambda=0.25$) on the increments of a surface with $H=0.5$.

*Source:* Reads `analysis_2d_all.json`, `paper_2d_mono.json` in `results/`.

In [ ]:
F.make_figure(9)

### Fig. 10. Daily returns: fluctuation functions and spectra

Daily returns. (a) Log returns of the S&P 500, 1950--2015. (b) $F_q(n)$ of backward MFQDMA ($\lambda=0.25$), shifted vertically. (c) $h(q)$ of the S&P 500 for MFDFA, MFDMA, and MFQDMA ($\lambda=0.1$, $0.25$, $0.5$); the dotted line marks $h=1/2$. (d) $f(\alpha)$ of the S&P 500, the SSE Composite, and Bitcoin for MFDMA and MFQDMA ($\lambda=0.25$).

*Source:* Reads the S&P 500 series from qrmdata (cloned into `data/` on first use) and `results/finance.json`.

In [ ]:
F.make_figure(10)

### Fig. 11. Eighteen assets

Eighteen assets. (a) Spectral width $\Delta\alpha$ of daily returns for MFDFA, MFDMA, and MFQDMA at five quantile levels. (b) Percentage of $\Delta\alpha$ retained by shuffled (open down-triangles) and phase-randomized (filled up-triangles) surrogates, mean of five each; colors as in (a).

*Source:* Reads `finance.json` in `results/`.

In [ ]:
F.make_figure(11)

### Fig. 12. Sensitivity to extremes and data errors

Sensitivity of $h(q)$ of daily returns (median $D_h$ over 18 assets). (a) Removal of the $k$ largest absolute returns. (b) Isolated bad ticks at a fraction $\varepsilon$ of days. (c) Erroneous returns (price level shifts) at a fraction $\varepsilon$ of days. (d) Microstructure noise of relative amplitude $\eta$ on every log price. Colors and markers as in panel (a).

*Source:* Reads `finance.json` in `results/`.

In [ ]:
F.make_figure(12)

### Fig. 13. DJIA stocks and rolling windows

(a), (b) Distribution of $D_h$ over 29 DJIA stocks (2000--2015) for 0.5% bad ticks and for microstructure noise with $\eta=0.5$; boxes show quartiles, whiskers the 5%--95% range; labels give the method or $\lambda$. (c) Change of $\Delta\alpha$ caused by 0.5% bad ticks in rolling ten-year windows of the S&P 500 (step: half a year).

*Source:* Reads `results/finance.json` and the S&P 500 series from qrmdata.

In [ ]:
F.make_figure(13)

### Fig. 14. Bonn EEG: surrogates

Bonn EEG, 20 segments per set. (a) Mean spectral width $\Delta\alpha$ of the originals. (b) Fraction of $\Delta\alpha$ retained by shuffled (open down-triangles) and phase-randomized (filled up-triangles) surrogates. (c) Mean fit quality $R^2$. Colors as in (a).

*Source:* Reads `results/real_eeg.json`.

In [ ]:
F.make_figure(14)

### Fig. 15. Bonn EEG: λ-resolved exponents

Median $\lambda$-resolved generalized Hurst exponents $h(q,\lambda)$ of the Bonn EEG sets (100 segments each, $\theta=0$, $12\le n\le128$). (a) Z, healthy, eyes open; (b) O, healthy, eyes closed; (c) N, interictal, contralateral; (d) F, interictal, epileptogenic zone; (e) S, ictal.

*Source:* Reads `results/eeg_main.npz`.

In [ ]:
F.make_figure(15)

### Fig. 16. Bonn EEG: information in λ

Information carried by the quantile level (500 EEG segments). (a) $\lambda$-spread $W(q=-5)$ per set. (b) Median $W(q)$ versus $q$. (c) Effect size $|\delta|$ (Cliff) of the F-versus-N contrast for each $(q,\lambda)$. (d) Cross-validated AUC of logistic classifiers using the 21 exponents $h(q)$ of MFDMA or the 21 exponents $h(q,\lambda^*)$ of MFQDMA at the best single $\lambda^*$ (equal feature dimension).

*Source:* Reads `results/eeg_main.npz` and `results/eeg_auc.json`.

In [ ]:
F.make_figure(16)

### Fig. 17. Terrain elevation model

Terrain elevation model. (a) Elevation $X$. (b) Mixed increments $\Delta_1\Delta_2X$. (c) $F_q(n)$ of 2D-MFQDMA ($\lambda=0.25$) on the increments, shifted vertically. (d) $f(\alpha)$ of the increments (solid), of shuffled increments (dotted, clustered near $\alpha\approx1$), and of a 2D phase-randomized surrogate (dashed), for 2D-MFDMA and 2D-MFQDMA.

*Source:* Computed directly (about 2 min). The elevation model ships with Matplotlib.

In [ ]:
F.make_figure(17)

## 3. Tables

Every data table of the paper is generated from `results/` by `mfqdma/tables_paperI.py`; the LaTeX body rows are written to `tables_out/`. Table I (the benchmark corpus) and Table XIII (guidelines) contain no computed numbers.

In [ ]:
from mfqdma import tables_paperI as T

### Table II

RMSE of $\tau(q)$ (mean ± s.d.) over the eight binomial measures.

In [ ]:
display(T.table2_binomial())

### Table III

RMSE of $h(q)$ versus $H$ (mean ± s.d.) over 18 fGn series ($N=2^{16}$).

In [ ]:
display(T.table3_fgn())

### Table IV

RMSE of $\tau(q)$ of the contaminated binomial measure ($p=0.3$, $N=2^{16}$, mean of five replicates).

In [ ]:
display(T.table4_contamination())

### Table V

RMSE of $\tau(q)$ of 2D cascades, backward windows.

In [ ]:
display(T.table5_cascade2d())

### Table VI

$h(2)$ of the mixed increments of fractional Brownian surfaces: mean of three realizations, standard deviation in units of the last digit.

In [ ]:
display(T.table6_mono2d())

### Table VII

Daily log returns of 18 assets: period, length, excess kurtosis, spectral widths, and the percentage of $\Delta\alpha$ retained by shuffled and phase-randomized surrogates.

In [ ]:
display(T.table7_markets())

### Table VIII

Sensitivity $D_h$ ($\times10^3$) of $h(q)$ of daily returns to genuine extremes and injected data errors (median over 18 assets or 29 DJIA stocks).

In [ ]:
display(T.table8_finrobust())

### Table IX

Sensitivity of the EEG exponents to preprocessing: mean $|\Delta h(q,1/2)|$ relative to the unprocessed data.

In [ ]:
display(T.table9_preprocessing())

### Table X

Bonn EEG, 20 segments per set: $\Delta\alpha$ of the originals and percentage retained by surrogates.

In [ ]:
display(T.table10_eeg())

### Table XI

Cross-validated AUC ($10\times10$ folds) of classifiers based on 21 exponents.

In [ ]:
display(T.table11_auc())

### Table XII

Terrain elevation model: raw surface, mixed increments and surrogates.

In [ ]:
display(T.table12_dem())

## 4. Check against the manuscript

Compares each regenerated figure (`figures_out/`) with the figure in the submitted manuscript (`figures/`). "identical" means pixel-identical PNG files, "close" that fewer than 1% of the pixels differ noticeably. The reference figures were drawn with Matplotlib 3.11 and its bundled DejaVu/Computer Modern fonts; with another Matplotlib version, small rendering differences (anti-aliasing, glyph placement) can make figures with identical data show up as "DIFFERS". The numbers themselves are checked by the tables in Part 3, which can be compared with the paper directly.

In [ ]:
from mfqdma.verify import compare_figures
display(compare_figures())

## 5. Download the outputs

In [ ]:
#@title Collect all regenerated figures, tables and results in one zip file
import shutil
from mfqdma import paths
stage = paths.REPO / '_export'
shutil.rmtree(stage, ignore_errors=True)
for d in ('figures_out', 'tables_out', 'results'):
    if (paths.REPO / d).is_dir():
        shutil.copytree(paths.REPO / d, stage / d)
archive = shutil.make_archive(str(paths.REPO / 'MFQDMA_PaperI_outputs'), 'zip', stage)
shutil.rmtree(stage)
print('wrote', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)